### Config

In [1]:
GPT_CONFIG_124M = {

    "vocab_size":      50257,
    "context_length":  1024,
    "emb_dim":         768,
    "n_layers":        12,
    "n_heads":         12,
    "drop_rate":       0.1,
    "qkv_bias":        False

}

### tokenizer

In [2]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
tokenizer

<Encoding 'gpt2'>

### download text - the-verdict

In [3]:
import urllib.request

url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt")

file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

len(text)

20479

### Datset
https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/dataloader.ipynb

In [4]:
from torch.utils.data import Dataset, DataLoader
import torch

class BookDataset(Dataset):

    def __init__(self, txt, tokenizer,stride, config):

        ctx_length = config['context_length']
        token_ids    = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        self.input_ids = []
        self.target_ids = []

        for i in range(0, len(token_ids) - ctx_length, stride):

            input_chunk  = token_ids[i: i + ctx_length]
            target_chunk = token_ids[i+1:i+ctx_length+1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, id):
        return self.input_ids[id], self.target_ids[id]

bds = BookDataset(text, tokenizer, stride=4, config =GPT_CONFIG_124M)
print("Ex.1 Inputs Shape: ",bds.input_ids[0].shape)
print("Ex.1 Targets Shape:",bds.target_ids[0].shape)

Ex.1 Inputs Shape:  torch.Size([1024])
Ex.1 Targets Shape: torch.Size([1024])


### DataLoaders

In [5]:
def create_dataloader(text, batch_size, stride, config, shuffle = True, drop_last = True, num_workers = 0):
    tokenizer  = tiktoken.get_encoding("gpt2")
    dataset    = BookDataset(text, tokenizer, stride, config)
    dataloader = DataLoader( dataset=dataset, 
                            batch_size=batch_size,
                            shuffle=shuffle,
                            drop_last=drop_last,
                            num_workers=num_workers, )

    return dataloader

dataloader = create_dataloader(text, batch_size=4, stride = 4, config = GPT_CONFIG_124M)

first_batch = next(iter((dataloader)))
print("1st Batch Inputs shape: ", first_batch[0].shape) 
print("1st Batch Targets shape:", first_batch[1].shape) 

1st Batch Inputs shape:  torch.Size([4, 1024])
1st Batch Targets shape: torch.Size([4, 1024])


### Masked Multi Attention

In [6]:
import torch.nn as nn

class MaskedMultiHeadAttn(nn.Module):
    
    def __init__(self, d_in, d_out, context_length, drop_rate, n_heads, qkv_bias = False):
        super().__init__()
        assert(d_out % n_heads == 0)

        self.d_in    = d_in
        self.d_out   = d_out
        self.n_heads = n_heads
        self.head_dim = d_out // n_heads
        

        self.W_query  = nn.Linear(d_in, d_out, qkv_bias)
        self.W_key    = nn.Linear(d_in, d_out, qkv_bias)
        self.W_value  = nn. Linear(d_in, d_out, qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout  = nn.Dropout(drop_rate)

        self.register_buffer("mask",torch.triu(torch.ones(context_length, context_length), diagonal =1))


    def forward(self, x):
        batch_size, seq_length, emb_dim = x.shape

        # shape: (batch_size, seq_length, d_in) --> (batch_size, seq_length, d_out)
        queries = self.W_query(x)  
        keys    = self.W_key(x)
        values  = self.W_value(x)

        # split d_out into n_heads & head_dim: (batch_size, seq_length, n_heads, head_dim)
        queries = queries.view(batch_size, seq_length, self.n_heads, self.head_dim) 
        keys    = keys.view(batch_size, seq_length, self.n_heads, self.head_dim) 
        values  = values.view(batch_size, seq_length, self.n_heads, self.head_dim) 

        # group by n_heads instead of seq_length: (batch_size, n_heads, seq_length, head_dim)
        queries = queries.transpose(1,2)
        keys    = keys.transpose(1,2)
        values  = values.transpose(1,2)

        # Attn Score: (batch_size, n_heads, seq_length, seq_length)
        attn_scores = queries @ keys.transpose(2,3)

        # Masking
        mask_bool = self.mask.bool()[:seq_length,:seq_length]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # Attn Weights: (batch_size, n_heads, seq_length, seq_length) 
        d_k = keys.shape[-1]
        attn_wts = torch.softmax(attn_scores / d_k **0.5, dim = -1)
        attn_wts = self.dropout(attn_wts)

        # context vector: (batch_size, n_heads, seq_length, head_dim)
        context_vecs = attn_wts @ values

        # context vector: (batch_size, seq_length, n_heads, head_dim)
        context_vecs = context_vecs.transpose(1,2)

        # context vector: (batch_size, seq_length, d_out)
        context_vecs = context_vecs.contiguous().view(batch_size, seq_length, self.d_out)

        context_vecs = self.out_proj(context_vecs)
        return context_vecs

In [7]:
# sample inputs - temp

b, seq_length       = first_batch[0].shape
emb_dim             = GPT_CONFIG_124M['emb_dim']

context_length      = GPT_CONFIG_124M['context_length']
drop_rate           = GPT_CONFIG_124M['drop_rate']
n_heads             = GPT_CONFIG_124M['n_heads']
qkv_bias            = GPT_CONFIG_124M['qkv_bias']

# sample embedding Layer
embeddings = nn.Embedding(GPT_CONFIG_124M['vocab_size'], GPT_CONFIG_124M['emb_dim'])
first_batch_inputs = embeddings(first_batch[0])

mma = MaskedMultiHeadAttn(emb_dim, emb_dim, context_length, drop_rate, n_heads, qkv_bias)
context_vec = mma(first_batch_inputs)
context_vec.shape

torch.Size([4, 1024, 768])

### Layer Norm

In [8]:
class LayerNorm(nn.Module):

    def __init__(self,emd_dim):

        super().__init__()
        self.eps   = 1e-5
        self.shift = nn.Parameter(torch.zeros(emd_dim))
        self.scale = nn.Parameter(torch.ones(emd_dim))

    def forward(self, x):
        mean   = x.mean(dim=-1, keepdim=True)
        var    = x.var(dim=-1, keepdim=True, unbiased = False)
        norm_x = (x - mean)/torch.sqrt(var + self.eps)

        return self.shift + self.scale * norm_x

### GELU

In [9]:
class GELU(nn.Module):
    
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

### FeedForward

In [10]:
class FeedForward(nn.Module):

    def __init__(self, emb_dim):
        
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            GELU(),
            nn.Linear(4 * emb_dim, emb_dim)
        )
        
    def forward(self, x):
        return self.layers(x)
        

In [ ]:
ffn = FeedForward(emb_dim=GPT_CONFIG_124M["emb_dim"])
output = ffn(context_vec)  # Calls forward(); shape: [4, 1024, 768]
output.shape

torch.Size([4, 1024, 768])

### Transformer

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.attn = MaskedMultiHeadAttn(
                        d_in  = cfg["emb_dim"],
                        d_out = cfg["emb_dim"],
                        context_length = cfg["context_length"],
                        drop_rate = cfg["drop_rate"],
                        n_heads   = cfg["n_heads"],
                        qkv_bias  = cfg["qkv_bias"])

        self.ff    = FeedForward(cfg["emb_dim"])
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut    = nn.Dropout(cfg["drop_rate"])

    
    def forward(self,x):

        shortcut = x
        x = self.norm1(x)
        x = self.attn(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

In [13]:
x = torch.rand([2,4,768])
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)
output.shape

torch.Size([2, 4, 768])

### GPT2

In [ ]:
class GPT2Model(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.tok_emb  = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
        self.pos_emb  = nn.Embedding(cfg['context_length'], cfg['emb_dim'])
        self.drop_emb = nn.Dropout(cfg['drop_rate']) 

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg['n_layers'])])

        self.final_norm = LayerNorm(cfg["emb_dim"])

        # nn.Linear always operates on the last dimension—this is built into its design. So you don’t need to specify dim=-1
        # [2, 4, emb_dim] → [2, 4, vocab_size]
        self.out_head = nn.Linear(cfg['emb_dim'],cfg['vocab_size'], bias = False)


    def forward(self, in_idx):
        
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device = in_idx.device)
        )
        
        x = tok_embeds + pos_embeds

        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

        return logits

### Text Generation

In [15]:
prompt = "My name is"
input_idx = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0)
print(input_idx)

input_idx.shape

tensor([[3666, 1438,  318]])


torch.Size([1, 3])

In [ ]:
def generate_text(model, idx, max_new_tokens, context_size ):

    for i in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:,-1,:]
        probas = torch.softmax(logits, dim = -1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

model = GPT2Model(GPT_CONFIG_124M)
model.eval()
out = generate_text(model = model,
                    idx = input_idx,
                    max_new_tokens = 6,
                    context_size = GPT_CONFIG_124M["context_length"])

out.shape


torch.Size([1, 9])

In [ ]:
def print_next_token(input_ids, model, tokenizer):
    logits = model(input_ids)
    next_token_logits = logits[:, -1, :]

    next_token_probs = torch.softmax(next_token_logits, dim=-1)
    next_token_ids = torch.argmax(next_token_probs, dim=-1)

    next_token_text = tokenizer.decode(next_token_ids.tolist())

    print("input_ids shape:", input_ids.shape)
    print("logits shape:", logits.shape)
    print("next_token_logits shape:", next_token_logits.shape)
    print("next_token_probs shape:", next_token_probs.shape)
    print("next_token_ids shape:", next_token_ids.shape)
    print("next_token_text:", next_token_text)

input_ids shape: torch.Size([1, 3])
logits shape: torch.Size([1, 3, 50257])
next_token_logits shape: torch.Size([1, 50257])
next_token_probs shape: torch.Size([1, 50257])
next_token_ids shape: torch.Size([1])
next_token_text: threat
